# MammoAI — ResNet18 Trenirovka (CBIS-DDSM)
GPU ni yoqing: Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Kaggle dataset yuklash
!pip install -q kaggle
from google.colab import files
print('kaggle.json faylini yuklang:')
files.upload()

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Dataset yuklab olish (~5GB, 5-10 daqiqa)
!kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset
!unzip -q cbis-ddsm-breast-cancer-image-dataset.zip -d /content/cbis-ddsm
print('Dataset yuklandi!')

In [ ]:
# 2. Dataset tayyorlash: cancer / normal papkalarga ajratish
import pandas as pd
import shutil
from pathlib import Path

DATASET = Path('/content/cbis-ddsm')
JPEG    = DATASET / 'jpeg'
DATA    = Path('/content/data')

LABEL_MAP = {
    'MALIGNANT': 'cancer',
    'BENIGN': 'normal',
    'BENIGN_WITHOUT_CALLBACK': 'normal'
}

for split in ('train', 'val'):
    for cls in ('normal', 'cancer'):
        (DATA / split / cls).mkdir(parents=True, exist_ok=True)

def find_img(partial):
    name = Path(partial).name
    for f in JPEG.rglob(name):
        return f
    return None

csvs = [
    ('csv/mass_case_description_train_set.csv', 'train'),
    ('csv/mass_case_description_test_set.csv',  'val'),
    ('csv/calc_case_description_train_set.csv', 'train'),
    ('csv/calc_case_description_test_set.csv',  'val'),
]

counts = {'train': {'normal':0,'cancer':0}, 'val': {'normal':0,'cancer':0}}

for csv_rel, split in csvs:
    csv_path = DATASET / csv_rel
    if not csv_path.exists(): continue
    df = pd.read_csv(csv_path)
    path_col = next((c for c in df.columns if 'pathology' in c.lower()), None)
    img_col  = next((c for c in df.columns if 'full mammogram' in c.lower()), None)
    if not path_col or not img_col: continue
    df = df.dropna(subset=[path_col, img_col])
    for _, row in df.iterrows():
        cls = LABEL_MAP.get(str(row[path_col]).strip().upper())
        if not cls: continue
        src = find_img(str(row[img_col]))
        if not src: continue
        n = counts[split][cls]
        shutil.copy2(src, DATA / split / cls / f'{split}_{cls}_{n:05d}{src.suffix}')
        counts[split][cls] += 1

print('Tayyor!')
for s in ('train','val'):
    print(f'  {s}: normal={counts[s]["normal"]}, cancer={counts[s]["cancer"]}')

In [ ]:
# 3. Model trenirovkasi
import torch, copy
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

train_ds = datasets.ImageFolder('/content/data/train', transform=transform)
val_ds   = datasets.ImageFolder('/content/data/val',   transform=transform)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
print('Klasslar:', train_ds.classes)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

best_acc, best_w = 0.0, None

for epoch in range(1, 26):
    model.train()
    loss_sum = 0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            correct += (out.argmax(1)==labels).sum().item()
            total += labels.size(0)

    val_acc = correct/total*100
    mark = ' ★' if val_acc > best_acc else ''
    print(f'Epoch {epoch:2d}/25 | Loss: {loss_sum/len(train_dl):.4f} | Val: {val_acc:.1f}%{mark}')
    if val_acc > best_acc:
        best_acc = val_acc
        best_w = copy.deepcopy(model.state_dict())

torch.save(best_w, '/content/model_breast.pth')
print(f'\nModel saqlandi! Val aniqlik: {best_acc:.1f}%')

In [ ]:
# 4. Modelni yuklab olish
from google.colab import files
files.download('/content/model_breast.pth')
print('model_breast.pth yuklab olindi!')
print('Uni  backend/model/model_breast.pth  ga qoyng')